In [1]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)
from utils.ica_pipeline import preprocess_for_ica, detrend_and_iir_bandpass
from utils.epoch_and_average import revise_annot

import mne

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
import seaborn as sns

import math
import re
from collections import namedtuple
from typing import Callable

from sklearn.model_selection import LeaveOneOut, KFold
from sklearn.linear_model import Ridge
import torch

from scipy.stats import ttest_1samp, mode
from statsmodels.stats.multitest import fdrcorrection

from collections import OrderedDict
from IPython.display import clear_output
import warnings

#### As a simplified example, assume
- SOA = 300 ms
- sesponse lag = 700 ms (i.e., each word's effect lasts 700 ms)
- sampling frequency = 10 Hz
- no intercept
- no log-transform
Then the <strong>finite impulse response / time-expanded deconvolution design matrix</strong> for Words 1-3 looks like<br><br>

100 ms (W1)&nbsp;&nbsp; [[1, 0, 0, 0, 0, 0, 0],<br>
200 ms &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[0, 1, 0, 0, 0, 0, 0],<br>
300 ms &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[0, 0, 1, 0, 0, 0, 0],<br>
400 ms (W2)&nbsp;&nbsp; [2, 0, 0, 1, 0, 0, 0],<br>
500 ms &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[0, 2, 0, 0, 1, 0, 0],<br>
600 ms &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[0, 0, 2, 0, 0, 1, 0],<br>
<strong>700 ms (W3)&nbsp;&nbsp;[3, 0, 0, 2, 0, 0, 1]</strong>,<br>
800 ms &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[0, 3, 0, 0, 2, 0, 0],<br>
900 ms &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[0, 0, 3, 0, 0, 2, 0]]<br><br>

To add an intercept, simply horizontally stack another 9-by-7 block matrix (to the left) of the matrix above, and replace every non-zero number with 1. Mathematically, this means there's a stable response throughout the entire time series regardless of word position.<br>

> <strong>REMARK 1.</strong> Notice how the row at 700 ms gets the effects of all Words 1-3.<br>

> <strong>REMARK 2.</strong> This is different in structure - though roughly similar in spirit - from "normal" rERP (regression-based event-related potentials) in that in Smith & Kutas, 2015, Brouwer et al., 2020, and Amanda Lin 2023 (no, I am not citing my never-published GitHub repo LOL), each row of the design matrix is a _trial_, and the regression can be computed at every single sampled time point by stacking the response vector temporally into a 2D matrix (and stacked into 3D for each channel). In contrast, a row in a time-expanded deconvolution design matrix is a _sampled time point_, and each predictor is itself a block matrix (instead of a column vector), then trials are vertically stacked.

The goal of this re-analysis pipeline is to show that adding the word position matrix block significantly improves model fit compared to intercept-only (by t-testing $\Delta R^{2}$). This is a <i>model-selection problem</i>, thus making nested CV (cross-validation) the most standard approach.<br><br>

Note also how this structure makes adding <i>frame valence</i> as a predictor not only conceptually (for me still), but also mathematically awkward:
- Suppose we take frame valence as a catecorgical variable (e.g., positive = -1, neutral = 0, negative = 1), then frame valence is identical across all words in that category regardless of position. This means for all words of frame valence x (x = -1, 0, 1), they have the exact same block matrix for the frame valence predictor, which then makes the intercept block matrix redundant as they are just scaled copies of each other. Vertically stacking trials from different frame valence categories might prevent literal rank deficiency / condition number blow-up, but structurally the design is redundant. So now we remove the global intercept, then the design matrix becomes: a different intercept for each frame valence category, but the same word position block matrix across all categories. This is not the "word position effect differs depending on valence" we're looking for.
- Adding frame valence by word position as an interaction is also not wise because even in standard OLS, interaction terms are known to cause collinearity. This is an even bigger concern for deconvolution since our SOA is constant with no jitters, and while Ridge helps with collinearity, it is not magic.
- Mathmatically we <i>could</i> add frame valence as a continuous predictor, which is the least likely to blow up rank / condition number. But this then becomes conceptually hard to interpret:
    - The core of finite-impulse response is to model the effect or response at a per-sampled-time-point level, not at a contexual, whole trial level.
    - Frame valence is fundamentally / essentially a metric that can only be derived <i>after</i> the participant has seen the second-to-last word of the setence. I find it logically awkward to retroactively "predict" online brain responses to Word 1 using a value that won't surface until Word 10.

$\Longrightarrow$<strong> Possible solution: Run the model separately for each frame valence category and test coefficient differences.</strong>
- Need to check sample size -- 80 trials per category per subject may be okay (Ridge is fitted and cross-validated within subject).

In [2]:
def build_design(*, soa: float, sfreq: float,
                 n_words: int, n_trials: int,
                 response_lag: float,
                 rank_tol: float=1e-3,
                 intercept: bool=True,
                 verbose: bool=False,
                 **predictors):
    if predictors:
        for p in predictors.values():
            assert p.shape == (n_trials, n_words), "Each predictor must be a numpy array of shape (n_trials, n_words)"

    DeconvDesignMatrix = namedtuple("DeconvDesignMatrix",
                                    ["predictors", "full", "n_lags", "rank", "singular_vals", "condition_num"])
    
    predictors = dict(predictors)
    if intercept:
        # predictors["intercept"] = np.ones((n_trials, n_words), dtype=float)
        predictors = {"intercept": np.ones((n_trials, n_words), dtype=float), **predictors}

    n_time_samples_per_trial = int(round(soa * n_words * sfreq))
    n_time_samples = int(n_time_samples_per_trial * n_trials) 
    n_lags = int(round(response_lag * sfreq))
    soa_samples = int(round(soa * sfreq))
    if verbose:
        print(f"# time samples per trial = {n_time_samples_per_trial}")
        print(f"response to each sampled time point lasts {n_lags} time points")
    
    matrices = {name: np.zeros((n_time_samples, n_lags), dtype=float) for name in predictors}
    
    positions = range(n_words)
    for i in range(n_trials):
        for j in positions:
            valid_range = min(n_time_samples_per_trial - j * soa_samples, n_lags)
            row = i * n_time_samples_per_trial + j * soa_samples
            for m, p in zip(matrices.values(), predictors.values()):
                m[row : row + valid_range, : valid_range] += p[i, j] * np.eye(valid_range)

    X_full = np.hstack([m for m in matrices.values()])
    rank = np.linalg.matrix_rank(X_full, tol=rank_tol)
    singular_vals = np.linalg.svd(X_full, compute_uv=False)
    condition_num = np.linalg.cond(X_full)

    return DeconvDesignMatrix(matrices, X_full, n_lags, rank, singular_vals, condition_num)

In [ ]:
WEIHUN_DIR = "/Users/jowanglin/regression-based_ERP/data/eeg/weihun"
CRYSTAL_DIR = "/Users/jowanglin/regression-based_ERP/data/eeg/crystal"
JY_DIR = "/Users/jowanglin/Word-Position-Effect_BLP-lab/Preprocessing/ica-data/SUBJ020"

warnings.filterwarnings("ignore")

# for JY's
def transform_description(arr: np.ndarray) -> np.ndarray:
    def helper(arr, idx):
        if arr[idx] == "241":
            return "241_pos" if int(arr[idx+1]) in range(1, 16) else "241_neg"
        elif arr[idx] == "242":
            return "242_pos" if int(arr[idx+1]) in range(31, 46) else "242_neg"
        elif arr[idx] == "243":
            return "243_pos" if int(arr[idx+1]) in range(61, 76) else "243_neg"
        elif arr[idx] == "244":
            return "244_pos" if int(arr[idx+1]) in range(91, 106) else "244_neg"
        else:
            return arr[idx]
                
    arr = [a.replace("S", "").strip() for a in arr]
    arr = [helper(arr, idx) for idx in range(len(arr))]
    return arr

num = 20
raw_file_name = f"subj{str(num).zfill(3)}.set"
raw = mne.io.read_raw_eeglab(f"{JY_DIR}/{raw_file_name}", verbose=False, preload=False)

annot = raw.annotations
df_annot = pd.DataFrame(annot)

fixation = "251"
non_final = "255"
pos = ("241_pos", "242_pos", "243_pos", "244_pos")
neu = ("245", "246", "247", "248")
neg = ("241_neg", "242_neg", "243_neg", "244_neg")

annot_revised = revise_annot(df_annot,
                             fixation=fixation,
                             non_final=non_final,
                             codes_after_fixation_are_final=False, 
                             transform_description=transform_description,
                             pos_frame=pos,
                             neu_frame=neu,
                             neg_frame=neg)
df_annot_revised = pd.DataFrame(annot_revised)
display(df_annot_revised.head(24))



,onset,duration,description,orig_time,extras
0,0.000,0.000,boundary,None,{}
1,34.625,0.001,254,None,{}
2,35.209,0.001,1,None,{}
3,45.816,0.001,254,None,{}
4,46.337,0.001,251/pos_frame,None,{}
5,47.354,0.001,w1/pos_frame,None,{}
6,47.871,0.001,w2/pos_frame,None,{}
7,48.404,0.001,w3/pos_frame,None,{}
8,48.921,0.001,w4/pos_frame,None,{}
9,49.437,0.001,w5/pos_frame,None,{}


In [100]:
raw.load_data()
raw_reref = preprocess_for_ica(raw.copy(),
                               ref_channels={"M2": 0.5},
                               return_raw_reref_only=True)
print(set(raw_reref.get_channel_types()))

# sanity check the reref is correct
data = raw.get_data()
eeg_ch_idx = [i for i, ch in enumerate(raw.ch_names) if "EO" not in ch]
print(np.array_equal((data - data[raw.ch_names.index("M2")]/2)[eeg_ch_idx],
                      raw_reref.get_data(picks="eeg")))

raw_reref_filt = detrend_and_iir_bandpass(raw_reref,
                                          l_freq=0.1, h_freq=30.0,
                                          order=2,
                                          ftype="butter")

standard_montage is not provided; defaulting to MNE-shipped standard_1020 and renaming all channel names to upper case.
{'eeg', 'eog'}
True
Creating RawArray with float64 data, n_channels=33, n_times=2682480
    Range : 0 ... 2682479 =      0.000 ...  2682.479 secs
Ready.


In [83]:
N_TRIALS = list(df_annot_revised["description"]).count("w1")
print(f"N_TRIALS = {N_TRIALS} trials")
SOA = mode(np.diff(df_annot_revised["onset"].to_numpy())).mode
print(f"SOA = {SOA} seconds")
DOWNSAMPLE_SFREQ = 100.0  
N_WORDS = 8

RIDGE_TOL = 1e-5
MAX_ITER = 20000
RANDOM_STATE = 42

N_TRIALS = 208 trials
SOA = 0.36599999999998545 seconds


In [84]:
word_pos_predictor = np.vstack([np.log(np.arange(1, N_WORDS+1)) for _ in range(N_TRIALS)])

matrix = build_design(soa=SOA,
                       sfreq=DOWNSAMPLE_SFREQ,
                       n_words=N_WORDS, n_trials=N_TRIALS,
                       response_lag=0.7,
                       rank_tol=1e-2,
                       intercept=True,
                       verbose=True,
                       word_pos=word_pos_predictor)
print(f"X_full.shape = {matrix.full.shape}")
print(f"rank = {matrix.rank}")
print(f"condition number = {matrix.condition_num}")
print(f"5 largest singular values: {matrix.singular_vals[:5]}")
print(f"5 smallest singular values: {matrix.singular_vals[-5:]}\n")

# time samples per trial = 293
response to each sampled time point lasts 70 time points
X_full.shape = (60944, 140)
rank = 140
condition number = 96.28808863571432
5 largest singular values: [94.89746972 94.89746972 94.89746972 94.89746972 94.89746972]
5 smallest singular values: [0.98555773 0.98555773 0.98555773 0.98555773 0.98555773]



In [68]:
class MySSP:
    # TODO: probably make it inherit from sklearn preprocessor or something
    # for better integration for future Amanda :D
    def __init__(self): 
        pass
    def fit_transform(self):
        pass
    def transform(self):
        pass
    

def tune_alpha(ridge_data: np.ndarray, *, X: np.ndarray,  
               alpha_range: np.ndarray,
               ssp: bool=False, info: mne.Info | None=None,
               roi_idx: list | None=None,
               loo: bool=False, n_splits: int | None=None,
               ridge_tol: float=1e-4, solver: str="auto", max_iter: int|None=None,
               score_on_mean: bool=True,
               verbose: bool=False):
    """ridge_data.shape = (n_trials, n_channels, n_time_samples); n_time_samples is total, not per trial
                          if doing LOO (e.g., training on 19 subjects, testing on 1), n_trials = n_subjects (but this does not work well)
       design matrix X.shape = (n_time_samples, n_predictors * n_lags)
       alpha_range in single-subject-level range (not divided by n_subj-1)
       if ssp is True, ridge_data should contain the full channel set needed for SSP, then only later restrict reporting / inference
       to ROI channels, where roi_idx must refer to the channel indices of the original full channel set.
       (If ridge_data contains only the ROI channels, then SSP is not very meaningful.)"""
    
    ScoreResults = namedtuple("ScoreResults", ["train_scores", "train_mean", "train_std",
                                               "valid_scores", "valid_mean", "valid_std"])
    CVResults = namedtuple("CVResults", ["fold_scores", "best_mean_score", "best_alpha"])   
    
    n_trials, n_channels, n_time_samples_per_trial = ridge_data.shape
    assert n_trials * n_time_samples_per_trial  == X.shape[0]
    X_split = np.stack(np.split(X, n_trials), axis=0)    # stack into an array so advance indexing works for train_idx and valid_idx
                                                         # X_split.shape (n_trials, n_time_samples_per_trial, n_lags)
    if roi_idx is None:
        print(f"ROI channel indices not set. Default to all channels in the input data.")
        roi_idx = list(range(n_channels))

    if loo:
        splitter = LeaveOneOut()
        n_splits = n_trials
    else:
        if n_splits is None:
            n_splits = 5
        splitter = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    
    train_scores = np.empty((n_splits, len(alpha_range), len(roi_idx)))
    valid_scores = np.empty((n_splits, len(alpha_range), len(roi_idx)))

    for i, (train_idx, valid_idx) in enumerate(splitter.split(ridge_data)):
        if verbose:
            print(f"  FOLD {i+1}")
         
        X_train_stack = np.vstack(X_split[train_idx])
        X_valid_stack = np.vstack(X_split[valid_idx])

        if score_on_mean:
            X_train_mean = np.mean(X_split[train_idx], axis=0)
            X_valid_mean = np.mean(X_split[valid_idx], axis=0)

        y_train, y_valid = ridge_data[train_idx], ridge_data[valid_idx]  # y_train.shape = (n_train_trials, n_channels, n_time_samples_per_trial)  
        y_train_stack = np.hstack([y for y in y_train])    # shape = (n_channels, n_train_trials * n_time_samples_per_trial)
        y_valid_stack = np.hstack([y for y in y_valid])    # shape = (n_channels, n_valid_trials * n_time_samples_per_trial)

        if ssp:
            my_ssp = MySSP(info=info)
            y_train_stack = my_ssp.fit_transform(y_train_stack)       
            y_valid_stack = my_ssp.transform(y_valid_stack)       

            y_train = np.stack(np.split(y_train_stack, len(train_idx), axis=1), axis=0)
            y_valid = np.stack(np.split(y_valid_stack, len(valid_idx), axis=1), axis=0)
            # y_train = y_train_stack.reshape(y_train.shape)  <- not safe
            # y_valid = y_valid_stack.reshape(y_valid.shape)  <- not safe    
        
        for j, alpha in enumerate(alpha_range):
            if verbose:
                print(f"ALPHA = {alpha}")   

            ridge = Ridge(alpha=alpha,
                          fit_intercept=False,         
                          tol=ridge_tol,
                          solver=solver,
                          positive=False,              
                          max_iter=max_iter,
                          random_state=RANDOM_STATE)
            
            for k, ch in enumerate(roi_idx):
                if verbose:
                    print(f"    Channel Index {ch}")

                ridge.fit(X_train_stack, y_train_stack[ch, :])

                if score_on_mean:
                    #X_train_mean = np.mean(X_split[train_idx], axis=0)  <- move outside of channel loop
                    #X_valid_mean = np.mean(X_split[valid_idx], axis=0)  <- move outside of channel loop
                    y_train_per_ch_mean = np.mean(y_train[:, ch, :], axis=0)
                    y_valid_per_ch_mean = np.mean(y_valid[:, ch, :], axis=0)
                    train_scores[i, j, k] = ridge.score(X_train_mean,
                                                        y_train_per_ch_mean)
                    valid_scores[i, j, k] = ridge.score(X_valid_mean,
                                                        y_valid_per_ch_mean)
                else:
                    train_scores[i, j, k] = ridge.score(X_train_stack,
                                                        y_train_stack[ch, :])
                    valid_scores[i, j, k] = ridge.score(X_valid_stack,
                                                        y_valid_stack[ch, :])
    
    fold_scores = {alpha: ScoreResults(train_scores[:, j, :], np.mean(train_scores[:, j, :]), np.std(train_scores[:, j, :]),
                                       valid_scores[:, j, :], np.mean(valid_scores[:, j, :]), np.std(valid_scores[:, j, :]))
                          for j, alpha in enumerate(alpha_range)}
            
    mean_scores = np.array([fold_scores[alpha].valid_mean for alpha in alpha_range])
    best_mean_score = np.max(mean_scores)
    best_alpha = alpha_range[np.argmax(mean_scores)]

    return CVResults(fold_scores, best_mean_score, best_alpha)

def nested_cv(subj_data: np.ndarray, *, X: np.ndarray,
              alpha_range: np.ndarray,
              ssp: bool=False, info: mne.Info | None=None,
              roi_idx: list | None=None,
              n_splits_outer: int=5, n_splits_inner: int=5,
              ridge_tol: float=1e-4, solver: str="auto", max_iter: int|None=None,
              score_on_mean: bool=True,
              verbose: bool=False):
    
    n_trials, n_channels, n_time_samples_per_trial = subj_data.shape
    assert n_trials * n_time_samples_per_trial  == X.shape[0]
    X_split_outer = np.stack(np.split(X, n_trials), axis=0)

    if roi_idx is None and verbose:
        print(f"ROI channel indices not set. Default to all channels in the input data.")
        roi_idx = list(range(n_channels))

    kf = KFold(n_splits=n_splits_outer, shuffle=True, random_state=RANDOM_STATE)

    outer_scores = np.empty((n_splits_outer, len(roi_idx)))
    best_alphas = np.empty((n_splits_outer, ))
    for i, (train_outer_idx, test_outer_idx) in enumerate(kf.split(subj_data)):
        if verbose:
            print(f"OUTER FOLD {i+1}")

        X_train_outer_stack = np.vstack(X_split_outer[train_outer_idx])
        X_test_outer_stack = np.vstack(X_split_outer[test_outer_idx])
        if score_on_mean:
            X_test_outer_mean = np.mean(X_split_outer[test_outer_idx], axis=0)
        
        y_train_outer = subj_data[train_outer_idx]
        y_test_outer = subj_data[test_outer_idx]

        cv_results = tune_alpha(y_train_outer, X=X_train_outer_stack,
                                alpha_range=alpha_range,
                                ssp=ssp, info=info,
                                roi_idx=roi_idx,
                                loo=False,
                                n_splits=n_splits_inner,
                                ridge_tol=ridge_tol,
                                score_on_mean=score_on_mean,
                                max_iter=max_iter,
                                solver=solver,
                                verbose=False)
        best_alphas[i] = cv_results.best_alpha

        ridge = Ridge(alpha=cv_results.best_alpha,
                      fit_intercept=False,         
                      tol=ridge_tol,
                      solver=solver,
                      positive=False,              
                      max_iter=max_iter,
                      random_state=RANDOM_STATE)
        
        y_train_outer_stack = np.hstack([y for y in y_train_outer])
        y_test_outer_stack = np.hstack([y for y in y_test_outer])

        if ssp:
            my_ssp = MySSP(info=info)
            y_train_outer_stack = my_ssp.fit_transform(y_train_outer_stack)
            y_test_outer_stack = my_ssp.transform(y_test_outer_stack)
            # y_train_outer = np.stack(np.split(y_train_outer_stack, len(train_outer_idx), axis=1), axis=0) <- not actually used, leaving it here for debugging
            y_test_outer = np.stack(np.split(y_test_outer_stack, len(test_outer_idx), axis=1), axis=0)

        for j, ch in enumerate(roi_idx):
            ridge.fit(X_train_outer_stack, y_train_outer_stack[ch, :])

            if score_on_mean:
                # X_test_outer_mean = np.mean(X_split_outer[test_outer_idx], axis=0)  <- move outside channel loop
                y_test_outer_per_ch_mean = np.mean(y_test_outer[:, ch, :], axis=0)
                outer_scores[i, j] = ridge.score(X_test_outer_mean,
                                                 y_test_outer_per_ch_mean)
            else:
                outer_scores[i, j] = ridge.score(X_test_outer_stack,
                                                 y_test_outer_stack[ch, :])

    return outer_scores, best_alphas




In [ ]:
warnings.resetwarnings()